# Clasificación de reportes ciudadanos PE-311 con BETO

**Tarea:** transfer learning (fine-tuning) de `dccuchile/bert-base-spanish-wwm-cased` sobre `descriptor_raw` para predecir `complaint_type` (8 clases).

**Pipeline:** carga → auditoría → limpieza → codificación de etiquetas → split estratificado → tokenización → fine-tuning → evaluación → inferencia → guardado.

> Para entrenar `priority` en lugar de `complaint_type`, cambia únicamente `TARGET_COL` en la celda de configuración. Todo lo demás (número de clases, pesos, métricas) se ajusta solo.


## 0. Entorno

In [ ]:
!pip install -q "transformers[torch]" datasets accelerate scikit-learn openpyxl

In [ ]:
import os, re, random, inspect, unicodedata
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: sin GPU el fine-tuning tardará horas. En Colab: Entorno de ejecución > Cambiar tipo de entorno > T4 GPU")

## 1. Configuración central

Todo lo que quieras tocar está aquí.

In [ ]:
# --- Datos ---
EXCEL_PATH   = "pe311_reportes_sinteticos.xlsx"
TEXT_COL     = "descriptor_raw"      # única entrada del modelo
TARGET_COL   = "complaint_type"      # cámbialo a "priority" para la otra tarea

# --- Modelo ---
MODEL_NAME   = "dccuchile/bert-base-spanish-wwm-cased"
MAX_LENGTH   = 128                   # se valida más abajo con la distribución real de tokens

# --- Entrenamiento ---
EPOCHS       = 4
LR           = 2e-5
BATCH_TRAIN  = 16
BATCH_EVAL   = 32
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
USE_CLASS_WEIGHTS = True             # compensa el desbalance de clases
OUTPUT_DIR   = "./beto_pe311"

# --- Split ---
TEST_SIZE = 0.15
VAL_SIZE  = 0.15                     # sobre el total

## 2. Carga del dataset

Si estás en Colab y el archivo no está en el directorio de trabajo, la celda te lo pedirá.

In [ ]:
if not os.path.exists(EXCEL_PATH):
    try:
        from google.colab import files
        print(f"No encuentro {EXCEL_PATH}. Súbelo:")
        uploaded = files.upload()
        EXCEL_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"Coloca {EXCEL_PATH} junto al notebook.")

ds = pd.read_excel(EXCEL_PATH)
print("Forma:", ds.shape)
ds.head(3)

## 3. Auditoría orientada a la tarea

No repetimos el EDA completo. Solo verificamos lo que puede romper el entrenamiento: nulos en las dos columnas que importan, duplicados de texto (fuga de datos entre train y test) y balance de clases.

In [ ]:
cols_utiles = [TEXT_COL, TARGET_COL]
print("--- Nulos en columnas de interés ---")
print(ds[cols_utiles].isnull().sum(), "\n")

print("--- Duplicados exactos de texto ---")
n_dup = ds[TEXT_COL].duplicated().sum()
print(f"{n_dup} de {len(ds)} ({n_dup/len(ds):.2%})\n")

print(f"--- Distribución de {TARGET_COL} ---")
dist = ds[TARGET_COL].value_counts()
print(dist)
print(f"\nRatio desbalance (mayoritaria / minoritaria): {dist.max()/dist.min():.2f}x")

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=dist.values, y=dist.index, ax=ax, color="#4C72B0")
ax.set_title(f"Distribución de {TARGET_COL}")
ax.set_xlabel("Nº de reportes")
plt.tight_layout(); plt.show()

In [ ]:
# Longitud del texto en caracteres, para dimensionar MAX_LENGTH
print(ds[TEXT_COL].str.len().describe())

### Columnas que NO deben usarse como features

Es tentador sumar más columnas al modelo, pero varias filtran la respuesta:

| Columna | Motivo del descarte |
|---|---|
| `descriptor` | Subcategoría (34 valores) anidada dentro de `complaint_type`. Determina la etiqueta. |
| `agency` | La entidad responsable se asigna a partir del tipo de reporte. Fuga directa. |
| `sla_dias` | Depende de la categoría y la prioridad. Fuga. |
| `closed_date`, `dias_resolucion`, `cumple_sla`, `status` | Información posterior al registro. En producción no existe al momento de clasificar. |
| `reporter_name`, `reporter_email`, `reporter_phone` | Datos personales sin señal, además de riesgo de privacidad. |

El escenario real es este: llega un texto libre y hay que clasificarlo. Por eso `descriptor_raw` es la única entrada legítima.

## 4. Limpieza y preprocesado del texto

Punto clave, porque aquí se equivoca casi todo el mundo con BERT: **no se hace el preprocesado clásico de NLP**. Nada de minúsculas, ni quitar stopwords, ni stemming, ni eliminar puntuación.

Razones concretas:

- El checkpoint es **cased**. Escribir en mayúsculas es señal ("URGENTE", nombres de avenidas). Bajar todo a minúsculas destruye información que el modelo sí sabe aprovechar.
- WordPiece ya maneja palabras raras partiéndolas en subtokens. El stemming solo genera fragmentos fuera de vocabulario.
- Las stopwords sostienen la sintaxis que la atención usa para relacionar términos.
- La puntuación delimita cláusulas.

Lo que sí corresponde: normalizar espacios en blanco, eliminar caracteres de control, descartar registros vacíos o demasiado cortos, y quitar duplicados exactos antes de partir el dataset.

In [ ]:
def limpiar_texto(t: str) -> str:
    """Normalización mínima y no destructiva para un modelo cased."""
    if not isinstance(t, str):
        return ""
    t = unicodedata.normalize("NFC", t)          # unifica representación de acentos
    t = t.replace("\u00a0", " ")                 # espacio duro
    t = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", t)   # caracteres de control
    t = re.sub(r"\s+", " ", t)                   # colapsa espacios, saltos y tabs
    return t.strip()

df = ds[[TEXT_COL, TARGET_COL]].copy()
n0 = len(df)

df[TEXT_COL] = df[TEXT_COL].apply(limpiar_texto)

df = df.dropna(subset=[TEXT_COL, TARGET_COL])
df = df[df[TEXT_COL].str.len() >= 20]                    # descarta ruido inservible
df = df.drop_duplicates(subset=[TEXT_COL], keep="first") # evita fuga train/test
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip().str.upper()
df = df.reset_index(drop=True)

print(f"Filas: {n0} -> {len(df)}  (se eliminaron {n0-len(df)})")
df.head(3)

In [ ]:
# Las clases con muy pocos ejemplos rompen la estratificación
conteo = df[TARGET_COL].value_counts()
raras = conteo[conteo < 10]
if len(raras):
    print("Clases con menos de 10 ejemplos, se descartan:\n", raras)
    df = df[~df[TARGET_COL].isin(raras.index)].reset_index(drop=True)
print("Clases finales:", df[TARGET_COL].nunique())

### Codificación de etiquetas

`LabelEncoder` fija el mapeo texto → entero. Guardamos `id2label` y `label2id` dentro del modelo para que las predicciones salgan ya con el nombre de la categoría.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label"] = le.fit_transform(df[TARGET_COL])

NUM_LABELS = len(le.classes_)
id2label = {i: str(c) for i, c in enumerate(le.classes_)}
label2id = {str(c): i for i, c in enumerate(le.classes_)}

print(f"NUM_LABELS = {NUM_LABELS}")
for i, c in id2label.items():
    print(f"  {i} -> {c}  ({(df['label']==i).sum()} ejemplos)")

## 5. Split estratificado train / validación / test

Tres particiones, no dos. La validación guía la selección del mejor checkpoint durante el entrenamiento; el test se toca una sola vez al final. Si evalúas sobre el mismo conjunto que usaste para elegir el checkpoint, la métrica que reportas está inflada.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[TEXT_COL].values
y = df["label"].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=TEST_SIZE + VAL_SIZE, stratify=y, random_state=SEED, shuffle=True
)
prop_test = TEST_SIZE / (TEST_SIZE + VAL_SIZE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=prop_test, stratify=y_temp, random_state=SEED, shuffle=True
)

print(f"Train: {len(X_train)}  |  Val: {len(X_val)}  |  Test: {len(X_test)}")

# Verificamos que la estratificación mantuvo las proporciones
comp = pd.DataFrame({
    "train": pd.Series(y_train).value_counts(normalize=True),
    "val":   pd.Series(y_val).value_counts(normalize=True),
    "test":  pd.Series(y_test).value_counts(normalize=True),
}).sort_index()
comp.index = [id2label[i] for i in comp.index]
(comp * 100).round(2)

## 6. Tokenización

El tokenizador de BETO se encarga de los tokens especiales `[CLS]` y `[SEP]`, del truncado y de la máscara de atención. Usamos **padding dinámico** por lote (`DataCollatorWithPadding`) en vez de `padding="max_length"`: rellenar todo a 128 cuando el 95 % de los textos ocupa la mitad es desperdiciar cómputo en tokens vacíos.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Verificamos empíricamente que MAX_LENGTH no está truncando información
muestra = df[TEXT_COL].sample(min(1000, len(df)), random_state=SEED).tolist()
longitudes = [len(tokenizer.encode(t)) for t in muestra]
print(pd.Series(longitudes).describe().round(1))
print(f"\nPercentil 99: {np.percentile(longitudes, 99):.0f} tokens")
print(f"Textos que superan MAX_LENGTH={MAX_LENGTH}: {np.mean(np.array(longitudes) > MAX_LENGTH):.2%}")

In [ ]:
from datasets import Dataset

def make_ds(textos, etiquetas):
    return Dataset.from_dict({"text": list(textos), "labels": [int(v) for v in etiquetas]})

ds_train = make_ds(X_train, y_train)
ds_val   = make_ds(X_val,   y_val)
ds_test  = make_ds(X_test,  y_test)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

ds_train = ds_train.map(tokenize_fn, batched=True, remove_columns=["text"])
ds_val   = ds_val.map(tokenize_fn,   batched=True, remove_columns=["text"])
ds_test  = ds_test.map(tokenize_fn,  batched=True, remove_columns=["text"])

print(ds_train)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Inspección de un lote real
lote = data_collator([ds_train[i] for i in range(4)])
print({k: tuple(v.shape) for k, v in lote.items()})

## 7. Modelo

`num_labels` sale del `LabelEncoder`, no de un número escrito a mano. Las advertencias de pesos `MISSING` para `classifier.weight`, `classifier.bias` y el `pooler` son esperadas y correctas: esa cabeza de clasificación se inicializa aleatoriamente y es justamente lo que vamos a entrenar. La transferencia ocurre en las 12 capas del encoder, que sí llegan preentrenadas.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParámetros entrenables: {entrenables:,}")

## 8. Pesos de clase

Con clases desbalanceadas la red aprende rápido que predecir la mayoritaria ya da buena accuracy. Los pesos inversos a la frecuencia hacen que un error en una clase minoritaria cueste más en la función de pérdida.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = None
if USE_CLASS_WEIGHTS:
    pesos = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_LABELS), y=y_train)
    class_weights = torch.tensor(pesos, dtype=torch.float)
    for i, w in enumerate(pesos):
        print(f"  {id2label[i]:<24} peso {w:.3f}")

## 9. Métricas

La accuracy engaña con datos desbalanceados. El criterio de selección del mejor checkpoint es **F1 macro**, que promedia las clases sin ponderar por tamaño y por tanto castiga el abandono de las minoritarias.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {
        "accuracy":    accuracy_score(labels, preds),
        "f1_macro":    f1,
        "precision_macro": p,
        "recall_macro":    r,
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

## 10. Trainer

Dos detalles prácticos:

- La API de `TrainingArguments` cambia entre versiones de `transformers` (`evaluation_strategy` pasó a llamarse `eval_strategy`, `Trainer(tokenizer=)` pasó a `processing_class=`). El constructor de abajo filtra los argumentos contra la firma real de tu versión instalada, así el notebook no se rompe en el peor momento del hackatón.
- `WeightedTrainer` sobrescribe `compute_loss` para inyectar los pesos de clase.

In [ ]:
from transformers import Trainer, TrainingArguments

def build_training_args(**kwargs):
    """Filtra kwargs según la firma real de TrainingArguments en esta versión."""
    firma = set(inspect.signature(TrainingArguments.__init__).parameters)
    if "eval_strategy" not in firma and "evaluation_strategy" in firma:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy", "epoch")
    validos = {k: v for k, v in kwargs.items() if k in firma}
    ignorados = set(kwargs) - set(validos)
    if ignorados:
        print("Argumentos no soportados por esta versión, se omiten:", ignorados)
    return TrainingArguments(**validos)

training_args = build_training_args(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    fp16=(device.type == "cuda"),
    report_to="none",
    seed=SEED,
)
print("TrainingArguments listo.")

In [ ]:
import torch.nn.functional as F

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = F.cross_entropy(logits.view(-1, model.config.num_labels), labels.view(-1), weight=w)
        return (loss, outputs) if return_outputs else loss

In [ ]:
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

# processing_class (nuevo) vs tokenizer (antiguo)
firma_trainer = set(inspect.signature(Trainer.__init__).parameters)
if "processing_class" in firma_trainer:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = WeightedTrainer(**trainer_kwargs)
print("Trainer listo.")

## 11. Entrenamiento

Con unos 4 000 ejemplos de entrenamiento y textos cortos, cada época ronda 1 a 2 minutos en una T4.

In [ ]:
train_result = trainer.train()
print("\n--- Resumen ---")
for k, v in train_result.metrics.items():
    print(f"{k}: {v}")

In [ ]:
hist = pd.DataFrame(trainer.state.log_history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

tr = hist[hist["loss"].notna()] if "loss" in hist else pd.DataFrame()
ev = hist[hist["eval_loss"].notna()] if "eval_loss" in hist else pd.DataFrame()

if len(tr):
    axes[0].plot(tr["epoch"], tr["loss"], label="train")
if len(ev):
    axes[0].plot(ev["epoch"], ev["eval_loss"], marker="o", label="val")
axes[0].set_title("Pérdida"); axes[0].set_xlabel("época"); axes[0].legend(); axes[0].grid(alpha=.3)

if len(ev):
    axes[1].plot(ev["epoch"], ev["eval_f1_macro"], marker="o", label="F1 macro")
    axes[1].plot(ev["epoch"], ev["eval_accuracy"], marker="s", label="Accuracy")
axes[1].set_title("Métricas de validación"); axes[1].set_xlabel("época"); axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

## 12. Evaluación final sobre el conjunto de test

Este conjunto no participó ni en el entrenamiento ni en la selección del checkpoint.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred_out = trainer.predict(ds_test)
logits = pred_out.predictions[0] if isinstance(pred_out.predictions, tuple) else pred_out.predictions
y_pred = np.argmax(logits, axis=-1)
y_true = pred_out.label_ids

print("--- Métricas globales ---")
for k, v in pred_out.metrics.items():
    if k.startswith("test_") and isinstance(v, float):
        print(f"{k}: {v:.4f}")

print("\n--- Reporte por clase ---")
print(classification_report(y_true, y_pred, target_names=[id2label[i] for i in range(NUM_LABELS)], digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize="true")
nombres = [id2label[i] for i in range(NUM_LABELS)]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=nombres, yticklabels=nombres,
            cbar_kws={"label": "proporción por fila"}, ax=ax)
ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
ax.set_title("Matriz de confusión normalizada por clase real")
plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# Los errores más informativos: qué confunde el modelo y con cuánta seguridad
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
err = pd.DataFrame({
    "texto": X_test,
    "real": [id2label[i] for i in y_true],
    "predicho": [id2label[i] for i in y_pred],
    "confianza": probs.max(axis=1),
})
err = err[err["real"] != err["predicho"]].sort_values("confianza", ascending=False)
print(f"Errores: {len(err)} de {len(y_true)} ({len(err)/len(y_true):.2%})\n")
pd.set_option("display.max_colwidth", 110)
err.head(10)

## 13. Inferencia

In [ ]:
def predecir(textos, top_k=3):
    if isinstance(textos, str):
        textos = [textos]
    textos = [limpiar_texto(t) for t in textos]   # misma limpieza que en entrenamiento
    enc = tokenizer(textos, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()

    salida = []
    for texto, fila in zip(textos, p):
        orden = np.argsort(fila)[::-1][:top_k]
        salida.append({
            "texto": texto,
            "prediccion": id2label[int(orden[0])],
            "confianza": float(fila[orden[0]]),
            "top_k": [(id2label[int(i)], round(float(fila[i]), 4)) for i in orden],
        })
    return salida


reportes = [
    "Hay un bache enorme en la avenida principal que puede causar un accidente.",
    "La farola de la esquina no enciende y la calle esta completamente a oscuras.",
    "Comunico una fuga de agua potable en la calzada de Jr. Puno cuadra 3, el agua corre desde ayer.",
    "Estimados, solicito la poda de los arboles del parque central, las ramas tocan los cables.",
]

for r in predecir(reportes):
    print(f"\n{r['texto']}")
    print(f"  -> {r['prediccion']}  ({r['confianza']:.1%})")
    for etq, pr in r["top_k"][1:]:
        print(f"     {etq}: {pr:.1%}")

## 14. Guardado

Se guardan modelo y tokenizador juntos. Con `id2label` dentro de la configuración, quien cargue el modelo obtiene las etiquetas legibles sin necesitar el `LabelEncoder`.

In [ ]:
RUTA_FINAL = f"{OUTPUT_DIR}/modelo_final"
trainer.save_model(RUTA_FINAL)
tokenizer.save_pretrained(RUTA_FINAL)
print("Guardado en:", RUTA_FINAL)

# Comprobación de carga
from transformers import pipeline
clf = pipeline("text-classification", model=RUTA_FINAL, tokenizer=RUTA_FINAL,
               device=0 if device.type == "cuda" else -1)
print(clf("El desmonte acumulado en la esquina lleva dos semanas sin recogerse."))

In [ ]:
# Descargar el modelo desde Colab (opcional)
# !zip -rq modelo_beto_pe311.zip {RUTA_FINAL}
# from google.colab import files; files.download("modelo_beto_pe311.zip")

## 15. Siguientes pasos

**Segunda tarea (`priority`).** Cambia `TARGET_COL = "priority"` y reejecuta desde la celda de configuración. Ojo con el desbalance: MEDIA concentra alrededor del 66 % de los registros, así que la accuracy sola no dirá nada útil. Ahí los pesos de clase y el F1 macro son imprescindibles.

**Un solo modelo para ambas tareas.** Si quieres predecir categoría y prioridad a la vez, lo eficiente es un encoder compartido con dos cabezas lineales y una pérdida sumada. Sale más barato en cómputo que dos BETO separados, aunque implica escribir un `nn.Module` propio en lugar de usar `AutoModelForSequenceClassification`.

**Línea base de comparación.** Antes de defender el modelo, entrena un TF-IDF con regresión logística sobre los mismos splits. Toma treinta segundos y es la referencia que te van a pedir: si BETO no la supera con claridad, el costo del transformer no se justifica.

**Si el F1 macro se estanca.** Prueba `EPOCHS=6` con `LR=3e-5`, o cambia el checkpoint por `PlanTL-GOB-ES/roberta-base-bne`, que suele rendir algo mejor en español.
